In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings 
from langchain_community.vectorstores import Chroma

/var/folders/cf/w8d5qyf95hbfw2kycgt8m1j80000gn/T/ipykernel_27984/2190460104.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
df = pd.read_csv("Data/cleaned_dataset.csv")

In [3]:
df.head()

,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
#Skapar text för embeddings 
df["text"] = (
    "Track name: " + df["track_name"].astype(str) +
    ". Artist: " + df["artists"].astype(str) +
    ". Album: " + df["album_name"].astype(str) +
    ". Genre: " + df["track_genre"].astype(str) +
    ". Popularity: " + df["popularity"].astype(str) +
    ". Danceability: " + df["danceability"].astype(str) +
    ". Energy: " + df["energy"].astype(str) +
    ". Tempo: " + df["tempo"].astype(str)
)

df["text"].head()

0    Track name: Comedy. Artist: Gen Hoshino. Album...
1    Track name: Ghost - Acoustic. Artist: Ben Wood...
2    Track name: To Begin Again. Artist: Ingrid Mic...
3    Track name: Can't Help Falling In Love. Artist...
4    Track name: Hold On. Artist: Chord Overstreet....
Name: text, dtype: str

In [5]:
# Skapar en lista av Document-objekt där varje objekt innehåller texten från "text"-kolumnen 
documents = []
for text in df["text"]:
    documents.append(Document(page_content=text))

len(documents)

113423

In [6]:
documents[0]

Document(metadata={}, page_content='Track name: Comedy. Artist: Gen Hoshino. Album: Comedy. Genre: acoustic. Popularity: 73. Danceability: 0.676. Energy: 0.461. Tempo: 87.917')

In [11]:
# Skapar en mindre lista av Document-objekt 
small_documents = documents[:10]
len(small_documents)

10

In [2]:
load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001"
)

ValidationError: 1 validation error for GoogleGenerativeAIEmbeddings
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'models/embedding-001'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

In [13]:
#skapar vecktordatabasen 
vectorstore = Chroma.from_documents(
    documents=small_documents,
    embedding=embeddings,
    persist_directory="chroma_db"
)

vectorstore.persist()

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}